# Schumann: live MEI editing and a small model experiment

Session 2, Block 2, **0–40 minutes**. Continue from the linking notebook;
keep the same local `schumann_edition_work` project and its page images.
The viewer reads the saved file. Edit its XML in VS Code (or the same file
in JupyterLab), save, and compare the new rendering with the source.

Model access is optional. The accompanying `vscode-free-models.md` records
free routes checked on **25 October 2026**; it is access research, not a model
benchmark. This notebook never contacts a model provider or asks for a key.
Prepare accounts/extensions/downloads before class. Start with one manual edit.

In [ ]:
%pip install -q "camat[validation,notebooks]==0.2.8" verovio==6.3.0

In [1]:
# Helper functions for this notebook. Run this cell; you do not need to read it.
# They follow the same rules as the demos on the course page.

"""Small local checks for the Schumann editing session; no model service calls."""

import hashlib
import json
from pathlib import Path
import xml.etree.ElementTree as ET

MEI_NS = "http://www.music-encoding.org/ns/mei"
XML_ID = "{http://www.w3.org/XML/1998/namespace}id"
M = "{" + MEI_NS + "}"
ET.register_namespace("", MEI_NS)


def editing_sha(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def audit_editing_changes(before, after):
    """Allow new dynam/tempo/slur/annot measure children; protect the rest.

    This is a structural guard, not a source-fidelity or schema validator.
    Whitespace-only formatting is ignored. Existing editorial/header elements
    remain protected. Every new top-level control needs a unique xml:id.
    """
    baseline = ET.parse(before).getroot()
    current = ET.parse(after).getroot()
    old_ids = {e.get(XML_ID) for e in baseline.iter() if e.get(XML_ID)}
    ids = [e.get(XML_ID) for e in current.iter() if e.get(XML_ID)]
    if len(ids) != len(set(ids)):
        raise ValueError("Duplicate xml:id in working MEI")
    if not old_ids.issubset(ids):
        raise ValueError("An existing xml:id was removed or renamed")
    allowed = {M + tag for tag in ("dynam", "tempo", "slur", "annot")}
    inserted = []

    def project(element, allow_insertions=False, parent=None):
        if (allow_insertions and parent is not None and parent.tag == M + "measure"
                and element.tag in allowed and element.get(XML_ID) not in old_ids):
            if not element.get(XML_ID):
                raise ValueError("Give each added control/annotation a unique xml:id")
            inserted.append({"id": element.get(XML_ID), "element": element.tag.split("}")[-1],
                             "measure": parent.get("n")})
            return None
        children = [value for child in element
                    if (value := project(child, allow_insertions, element)) is not None]
        return (element.tag, sorted(element.attrib.items()), (element.text or "").strip(),
                (element.tail or "").strip(), children)

    if project(baseline) != project(current, True):
        raise ValueError("Protected content changed: inspect the diff before continuing")
    graphics = current.findall("m:music/m:facsimile/m:surface/m:graphic", {"m": MEI_NS})
    for graphic in graphics:
        target = graphic.get("target", "")
        if not target or "://" in target or not (Path(after).parent / target).is_file():
            raise ValueError(f"Expected local graphic is missing or changed: {target}")
    return {"protected_structure_unchanged": True, "local_graphics_present": len(graphics),
            "new_controls": inserted, "before_sha256": editing_sha(before),
            "after_sha256": editing_sha(after), "human_source_review": "not_assessed"}


def forte_prompt(path):
    root = ET.parse(path).getroot()
    matches = [e for e in root.findall("m:music/m:body//m:measure", {"m": MEI_NS})
               if e.get(XML_ID) == "d1e1231"]
    if len(matches) != 1 or matches[0].get("n") != "6":
        raise ValueError("Expected measure d1e1231 / label 6 is missing")
    measure = matches[0]
    if any(e.get(XML_ID) == "seminar-dynam-second-system" for e in root.iter()):
        raise ValueError("The forte task is already applied; use the unedited checkpoint for a fair comparison")
    return '''Work on MEI 5.1, Schumann Op.68 No.10, edition/mei/work.mei.
Human-verified reading supplied for this exercise: IMSLP 51556, printed p.8,
second system, first full bar (musical bar 5 / MEI label 6), has a printed forte
at the start. Use this description as evidence; you have not inspected the scan.
Target: measure xml:id="d1e1231", n="6". Propose one <dynam> child after its
<staff> children: staff="1", place="below", tstamp="1", text f, unique
xml:id="seminar-dynam-second-system". Use ordinary transcription, not <supplied>.
Identify the insertion point, return only the minimal XML insertion and human
checks. Do not edit files, run commands, re-export or reformat. Preserve all
existing IDs, music, header, facsimile links and image paths. Ask if evidence
or targets are missing. Do not claim validation or personal source inspection.
Actual MEI excerpt (default namespace http://www.music-encoding.org/ns/mei):
''' + ET.tostring(measure, encoding="unicode")

In [2]:
import difflib
import subprocess
import sys
from IPython.display import display

PROJECT = (Path.cwd() / "schumann_edition_work").resolve()
WORK_MEI = PROJECT / "edition/mei/work.mei"
BEFORE_MEI = PROJECT / "edition/checkpoints/linked-before-editing.mei"
for path in (WORK_MEI, BEFORE_MEI):
    if not path.is_file():
        raise FileNotFoundError(f"Run the facsimile linking notebook first, or set PROJECT: {path}")
print("Edit this exact file:", WORK_MEI)
print("Unedited checkpoint:", BEFORE_MEI)
print("Working SHA-256:", editing_sha(WORK_MEI))
print(audit_editing_changes(BEFORE_MEI, WORK_MEI))

Edit this exact file: /tmp/camat-schumann-editing-verified/schumann_edition_work/edition/mei/work.mei
Unedited checkpoint: /tmp/camat-schumann-editing-verified/schumann_edition_work/edition/checkpoints/linked-before-editing.mei
Working SHA-256: ad594ea784895a4cb1ded45e9ca311049c346be26711de163b8de6bf7d200885
{'protected_structure_unchanged': True, 'local_graphics_present': 1, 'new_controls': [], 'before_sha256': 'ad594ea784895a4cb1ded45e9ca311049c346be26711de163b8de6bf7d200885', 'after_sha256': 'ad594ea784895a4cb1ded45e9ca311049c346be26711de163b8de6bf7d200885', 'human_source_review': 'not_assessed'}


## 0–10 — Start the live viewer

Run the next cell; keep its output visible beside the XML editor. Select
pickup C3 (`d1e60`) in the score; check the pickup source region. The second
system begins at measure `d1e1231`, MEI label 6 (musical bar 5).

Auto-watch follows this local path. Saving an unrelated local/cloud copy
cannot update it. If events are missed, use **Reload score** or relaunch with
`auto_watch_mode="always"`. HTTP inputs instead have **Reload source** and
cannot be watched. Trust the notebook if VS Code blocks widget scripts.
The saved course reference omits the image-bearing widget output/state.

In [3]:
from camat import launch_interactive_facsimile_viewer

if "viewer" in globals():
    viewer.stop_watch()
viewer = launch_interactive_facsimile_viewer(
    str(WORK_MEI), auto_watch_mei=True, auto_watch_mode="events",
    verovio_breaks="encoded", verovio_options={"header": "none", "footer": "none"},
    show_annotations=True, allow_missing_facsimile=False,
)
print("Live viewer launched; save", WORK_MEI.name, "in the XML editor.")

Live viewer launched; save work.mei in the XML editor.


## 10–20 — One manual restoration

Find pickup measure `d1e9`, label 1. Inspect the full first system: the source
prints **Frisch und munter.** and **f**. After its staff children, insert:

```xml
<dynam xml:id="seminar-dynam-opening" staff="1" place="below" tstamp="1">f</dynam>
<tempo xml:id="seminar-tempo-opening" staff="1" place="above" tstamp="1">Frisch und munter.</tempo>
```

Save and look for the watcher reporting **Reloaded**. Check the actual new
marks and source alignment; a reload message alone is not a source check.
Do not change the pickup duration or regenerate the linked MEI.

## 20–30 — Compare two small model proposals (optional)

The cell below prints one complete prompt with the actual second-system
measure. Give **the identical prompt** to two named models in fresh chats.
Keep each response. Start with Chat/proposal mode; manually apply one reviewed
insertion. Copilot Free's Auto can participate, with the model recorded as
unknown if it is not exposed. An account-free pair can do this manually.

This task asks for XML encoding from a supplied, source-verified reading;
it does not test optical recognition or prove that the model read music.
Inspect the scan yourself. For a fair rerun use the unedited checkpoint to
regenerate the prompt, rather than asking a second model after applying it.

In [4]:
PROMPT = forte_prompt(BEFORE_MEI)
print(PROMPT)

Work on MEI 5.1, Schumann Op.68 No.10, edition/mei/work.mei.
Human-verified reading supplied for this exercise: IMSLP 51556, printed p.8,
second system, first full bar (musical bar 5 / MEI label 6), has a printed forte
at the start. Use this description as evidence; you have not inspected the scan.
Target: measure xml:id="d1e1231", n="6". Propose one <dynam> child after its
<staff> children: staff="1", place="below", tstamp="1", text f, unique
xml:id="seminar-dynam-second-system". Use ordinary transcription, not <supplied>.
Identify the insertion point, return only the minimal XML insertion and human
checks. Do not edit files, run commands, re-export or reformat. Preserve all
existing IDs, music, header, facsimile links and image paths. Ask if evidence
or targets are missing. Do not claim validation or personal source inspection.
Actual MEI excerpt (default namespace http://www.music-encoding.org/ns/mei):
<measure xmlns="http://www.music-encoding.org/ns/mei" n="6" xml:id="d1e1231" facs

## 30–40 — Exact slur targets and an editorial explanation

In pickup measure `d1e9`, insert a slur from `#d1e60` to `#d1e338`, staff 2,
curvedir above; use new ID `seminar-slur-pickup`. In label 3, measure `d1e352`,
insert the separate slur from `#d1e561` to `#d1e634`, staff 2, curvedir above;
use ID `seminar-slur-bar2`. Confirm the actual label/ID in your file first.
The two printed phrases replace an inherited upstream slur that ran to C4
`d1e619`. The starter already corrects `d1e338` to F3; preserve that note.

Add an `<annot type="editorial">` in pickup measure `d1e9`, with a new unique
ID, a readable label and `plist="#d1e60 #d1e338 #d1e561 #d1e634"`. Use a `<p>`
explaining IMSLP 51556, printed p.8, the two slurs, the inherited discrepancy
and the incomplete scope. Optional prompt: `examples/model-prompts/annotate-slurs.md`.
Review any proposed prose. Printed slurs use ordinary transcription, not
`<supplied>` merely because the teaching starter omitted them.

## Inspect the saved diff and protected content

Run these cells again after edits. The local guard permits new measure-level
dynamic/tempo/slur/annotation elements with unique IDs and protects the other
XML content, including notes, metadata and facsimile. It ignores whitespace
formatting. It does not establish correct anchors, wording, source readings
or schema conformance; those checks follow separately.

In [5]:
print("".join(difflib.unified_diff(
    BEFORE_MEI.read_text().splitlines(keepends=True),
    WORK_MEI.read_text().splitlines(keepends=True),
    fromfile="linked-before-editing.mei", tofile="work.mei")))
AUDIT = audit_editing_changes(BEFORE_MEI, WORK_MEI)
print(json.dumps(AUDIT, indent=2))


{
  "protected_structure_unchanged": true,
  "local_graphics_present": 1,
  "new_controls": [],
  "before_sha256": "ad594ea784895a4cb1ded45e9ca311049c346be26711de163b8de6bf7d200885",
  "after_sha256": "ad594ea784895a4cb1ded45e9ca311049c346be26711de163b8de6bf7d200885",
  "human_source_review": "not_assessed"
}


In [6]:
VALIDATION_DIR = PROJECT / "edition/validation-editing"
validation = subprocess.run([
    sys.executable, "-m", "camat.mei_validation", str(WORK_MEI),
    "--output-dir", str(VALIDATION_DIR), "--verovio",
], text=True, capture_output=True, timeout=180)
print(validation.stdout)
if validation.returncode:
    print(validation.stderr)
    raise RuntimeError("Validation failed; inspect reports before accepting the patch")
run = json.loads((VALIDATION_DIR / "run.json").read_text())
print("Conformance:", run["conformance"])
print("Consumer diagnostics:", run["consumer_diagnostics"])
print("Human source comparison still needs your own record.")

MEI conformance: passed; consumers: incomplete

Conformance: passed
Consumer diagnostics: incomplete
Human source comparison still needs your own record.


## Save the experiment record

This creates a blank record once; it never invents model responses, passed
checks or human approval. Fill it in the editor, keeping prompts/responses and
patches as separate text files. Use a new filename for each model/run.
Existing records are preserved. The inherited copyright annotation is source
metadata; distinguish it from the new editorial annotation in the viewer.

In [7]:
RECORD = PROJECT / "edition/experiments/model-01.json"
if not RECORD.exists():
    RECORD.parent.mkdir(parents=True, exist_ok=True)
    template = {
        "status": "not_run", "date": "", "participant": "", "editor_version": "",
        "extension_and_version": "", "provider": "", "account_plan": "",
        "requested_model_id": "", "reported_model_id_or_unknown": "",
        "mode": "chat_proposal_manual_application", "task": "restore-forte",
        "input_mei_sha256": editing_sha(BEFORE_MEI), "prompt_file": "",
        "elapsed_seconds": None, "quota_before": "", "quota_after": "",
        "response_file": "", "patch_file": "", "unexpected_changes": [],
        "checks": {"xml_parse": "not_run", "protected_content_and_links": "not_run",
                   "mei_schema": "not_run", "references_and_resources": "not_run",
                   "verovio_render": "not_run", "human_source_comparison": "not_reviewed"},
        "human_reviewer": "", "human_decision_and_corrections": "",
    }
    with RECORD.open("x", encoding="utf-8") as output:
        output.write(json.dumps(template, indent=2) + "\n")
print("Fill in / preserve:", RECORD)

Fill in / preserve: /tmp/camat-schumann-editing-verified/schumann_edition_work/edition/experiments/model-01.json


Leave the viewer running while editing. At the end, run `viewer.stop_watch()`
(or turn off **Auto-watch MEI**). Next are the mei-friend alternative, peer
review and the publication build. No model call or source approval has been
performed merely by running this notebook.

## Instructor rehearsal only: deterministic save simulation

The student notebook above leaves editing to the participant. The following extra cells apply six **prepared teacher fragments on a scratch copy**, exercise both ordinary save and atomic replacement, and validate the result. These are not model responses or human source approval. The check observes native kernel/watch/widget state; VS Code frontend navigation is a separate rehearsal.

In [8]:
FRAGMENTS = [('d1e9', '<dynam xmlns="http://www.music-encoding.org/ns/mei" staff="1" place="below" tstamp="1" xml:id="seminar-dynam-opening">f</dynam>\n              '), ('d1e9', '<tempo xmlns="http://www.music-encoding.org/ns/mei" staff="1" place="above" tstamp="1" xml:id="seminar-tempo-opening">Frisch und munter.</tempo>\n              '), ('d1e1231', '<dynam xmlns="http://www.music-encoding.org/ns/mei" staff="1" place="below" tstamp="1" xml:id="seminar-dynam-second-system">f</dynam>\n            '), ('d1e9', '<slur xmlns="http://www.music-encoding.org/ns/mei" staff="2" curvedir="above" startid="#d1e60" endid="#d1e338" xml:id="seminar-slur-pickup"/>\n              '), ('d1e352', '<slur xmlns="http://www.music-encoding.org/ns/mei" staff="2" curvedir="above" startid="#d1e561" endid="#d1e634" xml:id="seminar-slur-bar2"/>\n            '), ('d1e9', '<annot xmlns="http://www.music-encoding.org/ns/mei" type="editorial" label="Opening slur division" plist="#d1e60 #d1e338 #d1e561 #d1e634" xml:id="seminar-annot-opening-slurs">\n                <p>IMSLP 51556, printed p. 8, first system: the pickup-to-F3 slur ends in the first full bar; a separate B-flat3-to-A3 slur occupies the second full bar. The base MEI instead joins the pickup to C4 in the second full bar. This exercise follows the selected print. Other missing expressive marks remain to be edited.</p>\n              </annot>\n            ')]
import asyncio
import os
import time
from lxml import etree as LX

async def wait_for_reload(previous):
    for _ in range(100):
        if viewer._render_revision > previous and not viewer._watch_state["reloading"]:
            return {"before_revision": previous, "after_revision": viewer._render_revision,
                    "status": viewer.watch_status.value}
        await asyncio.sleep(0.1)
    raise AssertionError("No completed watch-triggered render within 10 seconds")

def apply_fragments(fragments, atomic):
    tree = LX.parse(str(WORK_MEI))
    for measure_id, xml in fragments:
        matches = tree.xpath('//*[@xml:id=$id]', id=measure_id)
        assert len(matches) == 1
        matches[0].append(LX.fromstring(xml.encode()))
    data = LX.tostring(tree, encoding="UTF-8", xml_declaration=True)
    if atomic:
        staged = WORK_MEI.with_name(".rehearsal-save.mei")
        staged.write_bytes(data)
        os.replace(staged, WORK_MEI)
    else:
        WORK_MEI.write_bytes(data)

await asyncio.sleep(1.2)  # watcher startup settling; keep the kernel event loop active
revision = viewer._render_revision
apply_fragments(FRAGMENTS[:2], atomic=False)
direct_save = await wait_for_reload(revision)
await asyncio.sleep(1.2)  # respect the watcher's short post-reload suppression window
revision = viewer._render_revision
apply_fragments(FRAGMENTS[2:], atomic=True)
atomic_save = await wait_for_reload(revision)
AUDIT = audit_editing_changes(BEFORE_MEI, WORK_MEI)
assert len(AUDIT["new_controls"]) == 6
assert "seminar-dynam-second-system" in viewer.viewer_html.value
assert "seminar-slur-pickup" in viewer.viewer_html.value
assert "Opening slur division" in viewer.viewer_html.value
print("Direct save:", direct_save)
print("Atomic replacement:", atomic_save)
print("New dynamic/slur IDs and annotation label present in rebuilt viewer HTML.")


Direct save: {'before_revision': 1, 'after_revision': 2, 'status': "<b style='color:#047857'>Reloaded</b> at 16:47:12 (re-rendered score) · trigger=<code>closed</code> · mode=<code>events</code>"}
Atomic replacement: {'before_revision': 2, 'after_revision': 3, 'status': "<b style='color:#047857'>Reloaded</b> at 16:47:14 (re-rendered score) · trigger=<code>moved</code> · mode=<code>events</code>"}
New dynamic/slur IDs and annotation label present in rebuilt viewer HTML.


In [9]:
VALIDATION_DIR = PROJECT / "edition/validation-editing"
validation = subprocess.run([
    sys.executable, "-m", "camat.mei_validation", str(WORK_MEI),
    "--output-dir", str(VALIDATION_DIR), "--verovio",
], text=True, capture_output=True, timeout=180)
print(validation.stdout)
if validation.returncode:
    print(validation.stderr)
    raise RuntimeError("Validation failed; inspect reports before accepting the patch")
run = json.loads((VALIDATION_DIR / "run.json").read_text())
print("Conformance:", run["conformance"])
print("Consumer diagnostics:", run["consumer_diagnostics"])
print("Human source comparison still needs your own record.")

MEI conformance: passed; consumers: incomplete

Conformance: passed
Consumer diagnostics: incomplete
Human source comparison still needs your own record.


In [10]:
viewer.stop_watch()
negative_checks = {}
for name, xpath, attr, value in [
    ("pitch", './/*[@xml:id="d1e338"]', "pname", "g"),
    ("graphic_path", './/m:graphic', "target", "missing.png"),
    ("existing_header", './/m:meiHead', "type", "unexpected"),
]:
    mutated = LX.parse(str(WORK_MEI))
    targets = mutated.xpath(xpath, namespaces={"m": MEI_NS})
    assert targets, name
    targets[0].set(attr, value)
    candidate = WORK_MEI.with_name(f"rejected-{name}.mei")
    candidate.write_bytes(LX.tostring(mutated, encoding="UTF-8", xml_declaration=True))
    try:
        audit_editing_changes(BEFORE_MEI, candidate)
    except ValueError:
        negative_checks[name] = "rejected"
    else:
        raise AssertionError(f"Guard missed protected change: {name}")
    finally:
        candidate.unlink()
checks = {"prepared_on": "2026-10-25", "method": "deterministic teacher fragments on scratch copy; no model inference",
          "audit": AUDIT, "watch": {"direct_save": direct_save, "atomic_save": atomic_save,
          "new_content_in_viewer_html": True, "stopped_after_test": True},
          "negative_guard_checks": negative_checks,
          "validation": {key: run.get(key) for key in ("conformance", "required_checks", "consumer_diagnostics", "executions")},
          "human_source_approval": "not_asserted", "provider_accounts_and_inference": "not_tested",
          "vs_code_frontend_navigation": "not_tested"}
(PROJECT / "edition/editing-rehearsal.json").write_text(json.dumps(checks, indent=2) + "\n")
print(json.dumps(checks, indent=2))


{
  "prepared_on": "2026-10-25",
  "method": "deterministic teacher fragments on scratch copy; no model inference",
  "audit": {
    "protected_structure_unchanged": true,
    "local_graphics_present": 1,
    "new_controls": [
      {
        "id": "seminar-dynam-opening",
        "element": "dynam",
        "measure": "1"
      },
      {
        "id": "seminar-tempo-opening",
        "element": "tempo",
        "measure": "1"
      },
      {
        "id": "seminar-slur-pickup",
        "element": "slur",
        "measure": "1"
      },
      {
        "id": "seminar-annot-opening-slurs",
        "element": "annot",
        "measure": "1"
      },
      {
        "id": "seminar-slur-bar2",
        "element": "slur",
        "measure": "3"
      },
      {
        "id": "seminar-dynam-second-system",
        "element": "dynam",
        "measure": "6"
      }
    ],
    "before_sha256": "ad594ea784895a4cb1ded45e9ca311049c346be26711de163b8de6bf7d200885",
    "after_sha256": "0f9ecfec0a8